<a href="https://colab.research.google.com/github/Yesimg/IBM_Data_Science/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd

df= pd.read_excel('SAP_S4_Production_Planning_Dataset.xlsx')
df.head()

,Production_Order,Material_Number,Material_Description,Plant,MRP_Controller,Order_Qty,Due_Date,Order_Status,Material_Available,Critical_Component,Supplier_Name,Supplier_Delay_Days,Open_PO_Value_EUR,Work_Center,Planned_Hours,Actual_Hours,Schedule_Risk
0,500001,MAT-10001,Radar Assembly,IT02,A10,7,2026-07-15,Partially Confirmed,No,PCB Assembly,EuroElectronics,8,12143.88,CAL400,79.5,40.1,High
1,500002,MAT-10002,Sensor Package,IT02,B20,34,2026-09-23,Released,No,PCB Assembly,Global PCB,0,54969.77,ASSY100,50.1,62.8,High
2,500003,MAT-10003,Surveillance Module,IT01,A10,9,2026-05-28,Partially Confirmed,Yes,PCB Assembly,Global PCB,5,64212.37,ASSY100,53.9,123.5,Low
3,500004,MAT-10004,Radar Assembly,UK01,A10,29,2026-09-12,In Process,Yes,Cooling Unit,Thales Components,3,18807.76,TEST200,71.1,81.6,Low
4,500005,MAT-10005,Radar Assembly,IT02,B20,72,2026-08-05,Partially Confirmed,Yes,Antenna,RF Systems,7,34277.76,TEST200,61.3,73.0,Medium


In [12]:
#convert column names and dates
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.lower()

df['due_date'] = pd.to_datetime(df['due_date'])



**KPI 1: Total Orders**

In [13]:
total_orders = len(df)
print(f"Total Orders: {total_orders}")
#

Total Orders: 200


**KPI 2: Late Orders:**
orders due before today

In [14]:
from datetime import datetime

today=pd.Timestamp.today()

late_orders = df[df['due_date'] < today]
num_late_orders = len(late_orders)
print(f"Late Orders: {num_late_orders}")

Late Orders: 38


**KPI 3: High-Risk Orders:**

*   Material unavailable OR
*   Supplier delay > 10 days




In [20]:
high_risk= df[
 (df['material_available']=="No") |
 (df['supplier_delay_days'] > 10)
 ]

high_risk_count = len(high_risk)
print(f"High-Risk Orders: {high_risk_count}")

High-Risk Orders: 81


**DATA ANALYSIS**

In [22]:
!pip install plotly

In [24]:
def classify_risk(row):
    if row['supplier_delay_days'] > 10:
        return "High"
    elif row['supplier_delay_days'] > 5:
        return "Medium"
    else:
        return "Low"

df['risk_level'] = df.apply(classify_risk, axis=1)

In [25]:
risk_counts = df['risk_level'].value_counts()

risk_counts

,count
risk_level,
Low,112
Medium,52
High,36


In [26]:
import plotly.express as px

fig = px.bar(
    x=risk_counts.index,
    y=risk_counts.values,
    title="Orders by Risk Level"
)

fig.show()

In [28]:
fig = px.histogram(
    df,
    x='supplier_delay_days',
    title='Supplier Delay Distribution'
)

fig.show()

In [29]:
df['month'] = df['due_date'].dt.strftime('%Y-%m')

In [30]:
monthly = df.groupby('month').size().reset_index(name='orders')

In [32]:
fig = px.line(
    monthly,
    x='month',
    y='orders',
    title='Orders Due by Month'
)

fig.show()

In [35]:
#Dashboard
print("===== PRODUCTION DASHBOARD =====")

print(f"Total Orders: {total_orders}")
print(f"Late Orders: {num_late_orders}")
print(f"High Risk Orders: {high_risk_count}")

===== PRODUCTION DASHBOARD =====
Total Orders: 200
Late Orders: 38
High Risk Orders: 81
